# 遷移學習 (Transfer Learning)：站在預訓練模型的肩膀上

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/04-Transfer-Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_top5.py', 'q_dinov2.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"
import os
NUM_WORKERS = 2 if os.name != "nt" else 0   # Colab (Linux) 用 2 個 worker；Windows 本機請設 0

## 使用 ImageNet 預訓練的模型

`torchvision.models` 提供許多預訓練模型。新版的 API 用 `weights=` 指定權重，權重物件本身也附帶了前處理方式和類別名稱。

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights).to(device).eval()
preprocess = weights.transforms()
preprocess

In [ ]:
from urllib.request import urlopen
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
img = Image.open(urlopen(url)).convert("RGB")
display(img.resize((256, int(256 * img.height / img.width))))

In [ ]:
with torch.no_grad():
    logits = model(preprocess(img)[None].to(device))
probs = logits.softmax(1)[0]
idx = probs.argmax().item()
weights.meta["categories"][idx], probs[idx].item()

## Q
印出機率最高的 5 個類別和機率。（提示：`torch.topk`）

再試試看別的圖片。

In [ ]:
# %load q_top5.py
%run -i q_top5.py

## timm

[timm (PyTorch Image Models)](https://github.com/huggingface/pytorch-image-models) 收集了上千個影像模型與預訓練權重，Colab 已經預先安裝。

In [ ]:
import timm
timm.list_models("convnext*", pretrained=True)[:10]

## 資料集：Oxford-IIIT Pet

37 種貓狗品種，訓練資料只有 3680 張，平均每類 100 張左右。從頭訓練很難，但用預訓練模型就很容易。

這裡用 🤗 `datasets` 從 Hugging Face Hub 下載。

In [ ]:
from datasets import load_dataset
pets = load_dataset("timm/oxford-iiit-pet")
pets

In [ ]:
class_names = pets["train"].features["label"].names
ex = pets["train"][0]
print(class_names[ex["label"]])
display(ex["image"].resize((200, 200)))

## 建立模型

`timm.create_model(..., num_classes=37)` 會載入預訓練權重，並把最後的分類層換成新的 37 類（隨機初始化）。

In [ ]:
model_name = "convnext_tiny.fb_in22k_ft_in1k"
model = timm.create_model(model_name, pretrained=True, num_classes=len(class_names))
data_cfg = timm.data.resolve_model_data_config(model)
train_tf = timm.data.create_transform(**data_cfg, is_training=True)
test_tf = timm.data.create_transform(**data_cfg, is_training=False)
print(data_cfg)
print(f"參數數量: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
from torch.utils.data import DataLoader
def make_transform(tf):
    def fn(batch):
        return {"pixel_values": [tf(img.convert("RGB")) for img in batch["image"]], "label": batch["label"]}
    return fn
def collate(batch):
    return torch.stack([b["pixel_values"] for b in batch]), torch.tensor([b["label"] for b in batch])

train_ds = pets["train"].with_transform(make_transform(train_tf))
test_ds = pets["test"].with_transform(make_transform(test_tf))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=NUM_WORKERS, collate_fn=collate)
xb, yb = next(iter(train_loader))
xb.shape, yb[:10]

In [ ]:
@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for xb, yb in test_loader:
        with torch.autocast(device, dtype=torch.float16):
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return correct / len(test_ds)

def fit(model, params, epochs, lr):
    model.to(device)
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler(device)
    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            with torch.autocast(device, dtype=torch.float16):
                loss = F.cross_entropy(model(xb), yb, label_smoothing=0.1)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        print(f"epoch {epoch} loss={loss.item():.3f} test acc={evaluate(model):.4f} ({time.perf_counter()-t0:.0f}s)")

### 方法一：只訓練最後一層 (linear probing)

把 backbone 凍結 (`requires_grad=False`)，只訓練新的分類層。速度快、不容易 overfit。

In [ ]:
for p in model.parameters():
    p.requires_grad = False
head = model.get_classifier()
for p in head.parameters():
    p.requires_grad = True
print("可訓練參數:", sum(p.numel() for p in model.parameters() if p.requires_grad))
fit(model, head.parameters(), epochs=2, lr=1e-3)

### 方法二：整個模型一起微調 (fine-tuning)

用比較小的 learning rate 微調全部參數，通常可以再進步一些。

In [ ]:
for p in model.parameters():
    p.requires_grad = True
fit(model, model.parameters(), epochs=3, lr=1e-4)

In [ ]:
# 看看預測結果
model.eval()
for i in range(0, 3000, 500):
    ex = pets["test"][i]
    with torch.no_grad(), torch.autocast(device, dtype=torch.float16):
        pred = model(test_tf(ex["image"].convert("RGB"))[None].to(device)).argmax(1).item()
    display(ex["image"].resize((150, 150)))
    print("預測:", class_names[pred], " 答案:", class_names[ex["label"]])

## Q：自監督模型 DINOv2 + 傳統機器學習

[DINOv2](https://huggingface.co/facebook/dinov2-small) 是用自監督學習 (self-supervised learning) 訓練的 Vision Transformer，**完全沒看過標籤**，但它的特徵非常好用。

* 用 `transformers` 的 `AutoImageProcessor`, `AutoModel` 載入 `facebook/dinov2-small`
* 對每張圖取 `pooler_output`（CLS token 的特徵）
* 用 scikit-learn 的 `LogisticRegression` 訓練分類器，看看 test accuracy

（下一個單元會正式介紹 🤗 transformers）

In [ ]:
# %load q_dinov2.py
%run -i q_dinov2.py